# Notebook 09: Quantile Treatment Effects (QTE)

**Level**: Very Advanced | **Duration**: ~180 min | **Prerequisites**: Notebooks 01-03, 07, causal inference basics

## Learning Objectives

1. Understand quantile treatment effects vs average treatment effects
2. Estimate conditional QTE (standard quantile regression)
3. Implement unconditional QTE using RIF regression (Firpo et al. 2009)
4. Apply Difference-in-Differences Quantile Regression (DiD-QR)
5. Use Changes-in-Changes method (Athey & Imbens 2006)
6. Interpret heterogeneous treatment effects across the distribution

**Dataset**: Labor program evaluation (~2,000 observations)

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# PanelBox imports
from panelbox.models.quantile import PooledQuantile
from panelbox.models.quantile.treatment_effects import QuantileTreatmentEffects

# Matplotlib configuration
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)
np.random.seed(42)

# Paths
BASE_DIR = Path("..")
OUTPUT_DIR = BASE_DIR / "outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 1. Introduction & Motivation

### Research Question

> *"Does a job training program help all workers equally, or does it benefit low-wage vs high-wage workers differently?"*

The **Average Treatment Effect (ATE)** answers: "What is the mean impact?" But this single number can hide crucial heterogeneity.

**Quantile Treatment Effects (QTE)** reveal *how treatment effects vary across the outcome distribution*:

```
AVERAGE TREATMENT EFFECT (ATE):
  ATE = E[Y(1) - Y(0)]
  → Single number: Average impact across all individuals

QUANTILE TREATMENT EFFECT (QTE):
  QTE(τ) = Q_τ(Y(1)) - Q_τ(Y(0))
  → Function of τ: Impact varies across distribution

EXAMPLE — Job training program:
  ATE = $500/month → "Average worker gains $500"

  BUT QTE reveals:
  QTE(0.1) = $800  → Low earners gain MORE
  QTE(0.5) = $500  → Median worker gains average
  QTE(0.9) = $200  → High earners gain LESS

  → Program is PROGRESSIVE (helps poor more than rich)
     ATE alone would miss this!
```

In [ ]:
# Load labor program evaluation data
from panelbox.datasets import load_dataset

data = load_dataset("labor_program", category="quantile")

print(f"Dataset shape: {data.shape}")
print(f"\nVariables: {list(data.columns)}")
print("\nTreatment distribution:")
print(data["treatment"].value_counts())
print("\nPre/post periods:")
print(data["period"].value_counts().rename({0: "Pre (period=0)", 1: "Post (period=1)"}))
print("\nSummary statistics:")
display(data.describe().round(2))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Earnings distribution by treatment status
for treat, label, color in [(0, "Control", "steelblue"), (1, "Treated", "#D55E00")]:
    subset = data[data["treatment"] == treat]["earnings"]
    axes[0].hist(subset, bins=30, alpha=0.5, label=label, color=color, density=True)
axes[0].set_xlabel("Earnings ($)", fontsize=12)
axes[0].set_ylabel("Density", fontsize=12)
axes[0].set_title("Panel A: Earnings by Treatment Status", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Panel B: Earnings by period for treated vs control
for treat, label, color in [(0, "Control", "steelblue"), (1, "Treated", "#D55E00")]:
    for period, ls in [(0, "--"), (1, "-")]:
        subset = data[(data["treatment"] == treat) & (data["period"] == period)]["earnings"]
        period_label = "Post" if period == 1 else "Pre"
        axes[1].hist(subset, bins=25, alpha=0.3, density=True, color=color, linestyle=ls)
means = data.groupby(["treatment", "period"])["earnings"].mean().unstack()
axes[1].set_xlabel("Earnings ($)", fontsize=12)
axes[1].set_title("Panel B: Earnings by Treatment x Period", fontsize=13, fontweight="bold")
axes[1].grid(alpha=0.3)

# Panel C: ATE vs potential heterogeneity
treated_post = data[(data["treatment"] == 1) & (data["period"] == 1)]["earnings"]
control_post = data[(data["treatment"] == 0) & (data["period"] == 1)]["earnings"]
quantiles_plot = np.arange(0.05, 1.0, 0.05)
q_treated = np.quantile(treated_post, quantiles_plot)
q_control = np.quantile(control_post, quantiles_plot)
axes[2].plot(
    quantiles_plot,
    q_treated - q_control,
    "o-",
    color="purple",
    linewidth=2,
    markersize=4,
    label="Quantile difference",
)
ate_simple = treated_post.mean() - control_post.mean()
axes[2].axhline(
    ate_simple, color="red", linestyle="--", linewidth=2, label=f"ATE = ${ate_simple:.0f}"
)
axes[2].set_xlabel("Quantile (\u03c4)", fontsize=12)
axes[2].set_ylabel("Difference ($)", fontsize=12)
axes[2].set_title("Panel C: Quantile Differences (Post Period)", fontsize=13, fontweight="bold")
axes[2].legend(fontsize=10)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Theoretical Concepts

### 2.1 Treatment Effects Framework

**Potential Outcomes (Rubin Causal Model)**:

| Concept | Notation | Meaning |
|---------|----------|---------|
| Potential outcome (treated) | Y(1) | Outcome if individual receives treatment |
| Potential outcome (control) | Y(0) | Outcome if individual does NOT receive treatment |
| Observed outcome | Y = D\u00b7Y(1) + (1-D)\u00b7Y(0) | What we actually observe |
| Individual TE | \u03c4\u1d62 = Y\u1d62(1) - Y\u1d62(0) | NOT observable (fundamental problem) |
| ATE | E[Y(1) - Y(0)] | Average across population |

### Key Insight: ATE vs QTE

```
ATE = E[Y(1)] - E[Y(0)]
    → A single number summarizing the "average" effect

QTE(τ) = Q_τ(Y(1)) - Q_τ(Y(0))
    → A FUNCTION showing how effects vary across the distribution
    → Reveals heterogeneity that ATE hides
```

### 2.2 Conditional vs Unconditional QTE

Two fundamentally different concepts:

**Conditional QTE (CQTE)** --- Standard quantile regression approach:
```
CQTE(τ|X) = Q_τ(Y|D=1, X) - Q_τ(Y|D=0, X)

Model: Q_τ(Y|D,X) = X'β(τ) + δ(τ)·D
CQTE = δ(τ) = coefficient on treatment indicator

Interpretation: Effect on the τ-quantile of Y CONDITIONAL on X
→ "Among workers with same education/age, how does treatment
   affect the 10th percentile earner vs the 90th percentile earner?"
```

**Unconditional QTE (UQTE)** --- RIF regression (Firpo, Fortin, Lemieux 2009):
```
UQTE(τ) = Q_τ(Y(1)) - Q_τ(Y(0))

Uses Recentered Influence Function:
RIF(Y; Q_τ) = Q_τ + (τ - I(Y ≤ Q_τ)) / f_Y(Q_τ)

Interpretation: Effect on the τ-quantile of the POPULATION distribution
→ "How does treatment shift the 10th percentile of the overall
   earnings distribution?"
```

**When do they differ?**
- CQTE ≠ UQTE when covariates X are correlated with treatment effects
- CQTE conditions on X; UQTE integrates over X distribution
- For policy: UQTE is often more relevant (population-level effects)

### 2.3 Difference-in-Differences QR (DiD-QR)

For **panel data** with pre/post treatment:

```
Two periods: t=0 (pre-treatment), t=1 (post-treatment)
Two groups: D=1 (treated), D=0 (control)

DiD-QR estimator:
QTE(τ) = [Q_τ(Y|D=1,t=1) - Q_τ(Y|D=1,t=0)] 
        - [Q_τ(Y|D=0,t=1) - Q_τ(Y|D=0,t=0)]

Key assumption: PARALLEL TRENDS IN QUANTILES
→ In absence of treatment, the τ-quantile of treated group would 
  have evolved like control group's τ-quantile
```

### 2.4 Changes-in-Changes (Athey & Imbens 2006)

**Relaxes parallel trends**:

```
Instead of parallel trends in quantiles, CiC assumes:
- Rank invariance: Individual's rank in the distribution is stable
- Production function monotonicity

Estimator uses quantile function transformation:
Q_τ(Y(0)|D=1,t=1) = F⁻¹_{Y|D=0,t=1}(F_{Y|D=0,t=0}(Q_τ(Y|D=1,t=0)))

Counterfactual: "Where would treated group be if they experienced
the same distributional changes as the control group?"

QTE_CiC(τ) = Q_τ(Y|D=1,t=1) - Q_τ(Y(0)|D=1,t=1)
```

## 3. Implementation

We now implement all four QTE methods using PanelBox's `QuantileTreatmentEffects` class.

In [ ]:
# ============================================================
# METHOD 1: Conditional QTE via Standard Quantile Regression
# ============================================================

tau_grid = [0.1, 0.25, 0.5, 0.75, 0.9]
var_names = ["const", "treatment", "education", "age", "experience"]

# Build design matrix (PooledQuantile takes arrays, not formulas)
y = data["earnings"].values
X = np.column_stack(
    [
        np.ones(len(data)),
        data["treatment"].values,
        data["education"].values,
        data["age"].values,
        data["experience"].values,
    ]
)

cqte_results = {}
cqte_se = {}

print("CONDITIONAL QTE (Standard Quantile Regression)")
print("=" * 55)
print(f"{'\u03c4':<6} {'CQTE':>10} {'Std Err':>10} {'t-stat':>8} {'p-value':>8}")
print("-" * 55)

for tau in tau_grid:
    try:
        model = PooledQuantile(y, X, entity_id=data["id"].values, quantiles=tau)
        result = model.fit(se_type="cluster")

        # Treatment coefficient is index 1
        cqte = result.params.ravel()[1]
        se = result.std_errors.ravel()[1]
        t_stat = cqte / se
        p_val = 2 * (1 - stats.norm.cdf(abs(t_stat)))

        cqte_results[tau] = cqte
        cqte_se[tau] = se

        sig = "***" if p_val < 0.01 else ("**" if p_val < 0.05 else ("*" if p_val < 0.1 else ""))
        print(f"{tau:<6.2f} {cqte:>10.2f} {se:>10.2f} {t_stat:>8.2f} {p_val:>8.4f} {sig}")
    except Exception as e:
        print(f"\u03c4={tau:.2f}: Error - {e}")
        cqte_results[tau] = np.nan
        cqte_se[tau] = np.nan

print(
    "\nInterpretation: \u03b4(\u03c4) = effect of treatment on the \u03c4-th conditional quantile"
)

### Method 2: Unconditional QTE via RIF Regression

The RIF approach estimates the effect of treatment on the **unconditional** (population) quantiles, rather than conditional quantiles.

In [ ]:
# ============================================================
# METHOD 2: Unconditional QTE via RIF Regression
# ============================================================

qte_model_rif = QuantileTreatmentEffects(
    data=data,
    outcome="earnings",
    treatment="treatment",
    covariates=["education", "age", "experience"],
    entity_col="id",
    time_col="period",
)

try:
    rif_result = qte_model_rif.estimate_qte(tau=np.array(tau_grid), method="unconditional")

    print("UNCONDITIONAL QTE (RIF Regression)")
    rif_result.summary()

    # Also show as dataframe
    print("\nResults as DataFrame:")
    display(rif_result.to_dataframe().round(2))
except Exception as e:
    print(f"RIF estimation error: {e}")
    import traceback

    traceback.print_exc()

### Method 3: Difference-in-Differences QR

Using panel structure (pre/post periods) to estimate treatment effects via DiD at each quantile.

In [ ]:
# ============================================================
# METHOD 3: Difference-in-Differences QR
# ============================================================

qte_model_did = QuantileTreatmentEffects(
    data=data,
    outcome="earnings",
    treatment="treatment",
    covariates=["education", "age", "experience"],
    entity_col="id",
    time_col="period",
)

try:
    did_result = qte_model_did.estimate_qte(
        tau=np.array(tau_grid),
        method="did",
        n_boot=199,  # Fewer bootstraps for speed in tutorial
    )

    print("DIFFERENCE-IN-DIFFERENCES QR")
    did_result.summary()

    print("\nResults as DataFrame:")
    display(did_result.to_dataframe().round(2))
except Exception as e:
    print(f"DiD-QR estimation error: {e}")
    import traceback

    traceback.print_exc()

### Method 4: Changes-in-Changes (Athey & Imbens 2006)

CiC relaxes the parallel trends assumption by using the entire distribution transformation, not just quantile-level comparisons.

In [ ]:
# ============================================================
# METHOD 4: Changes-in-Changes
# ============================================================

qte_model_cic = QuantileTreatmentEffects(
    data=data, outcome="earnings", treatment="treatment", entity_col="id", time_col="period"
)

try:
    cic_result = qte_model_cic.estimate_qte(tau=np.array(tau_grid), method="cic")

    print("CHANGES-IN-CHANGES (Athey & Imbens 2006)")
    cic_result.summary()

    print("\nResults as DataFrame:")
    display(cic_result.to_dataframe().round(2))
except Exception as e:
    print(f"CiC estimation error: {e}")
    import traceback

    traceback.print_exc()

### 3.5 Comparison of All Methods

Now we compare CQTE, UQTE (RIF), DiD-QR, and CiC estimates side by side.

In [ ]:
# ============================================================
# COMPARISON OF ALL QTE METHODS
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# --- Panel A: QTE estimates across methods ---

# CQTE
cqte_vals = [cqte_results.get(tau, np.nan) for tau in tau_grid]
ax1.plot(
    tau_grid,
    cqte_vals,
    "o-",
    label="CQTE (Standard QR)",
    linewidth=2.5,
    markersize=8,
    color="steelblue",
)

# UQTE (RIF)
try:
    rif_vals = [rif_result.qte_results[tau]["qte"] for tau in tau_grid]
    ax1.plot(
        tau_grid, rif_vals, "s-", label="UQTE (RIF)", linewidth=2.5, markersize=8, color="#009E73"
    )
except:
    pass

# DiD-QR
try:
    did_vals = [did_result.qte_results[tau]["qte"] for tau in tau_grid]
    ax1.plot(tau_grid, did_vals, "^-", label="DiD-QR", linewidth=2.5, markersize=8, color="#D55E00")
except:
    pass

# CiC
try:
    cic_vals = [cic_result.qte_results[tau]["qte"] for tau in tau_grid]
    ax1.plot(tau_grid, cic_vals, "D-", label="CiC", linewidth=2.5, markersize=8, color="#CC79A7")
except:
    pass

# ATE reference line
ate = (
    data[data["treatment"] == 1]["earnings"].mean()
    - data[data["treatment"] == 0]["earnings"].mean()
)
ax1.axhline(ate, color="red", linestyle="--", linewidth=2, alpha=0.7, label=f"ATE = ${ate:.0f}")
ax1.axhline(0, color="black", linestyle=":", alpha=0.4)

ax1.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Treatment Effect ($)", fontsize=12, fontweight="bold")
ax1.set_title("Panel A: Comparison of QTE Methods", fontsize=14, fontweight="bold")
ax1.legend(fontsize=10, loc="best")
ax1.grid(alpha=0.3)

# --- Panel B: DiD-QR with confidence intervals ---
try:
    did_qte_vals = [did_result.qte_results[tau]["qte"] for tau in tau_grid]
    did_se_vals = [did_result.qte_results[tau]["se"] for tau in tau_grid]
    did_ci_lo = [did_result.qte_results[tau]["ci_lower"] for tau in tau_grid]
    did_ci_hi = [did_result.qte_results[tau]["ci_upper"] for tau in tau_grid]

    ax2.plot(
        tau_grid,
        did_qte_vals,
        "o-",
        linewidth=2.5,
        markersize=8,
        color="purple",
        label="DiD-QR Estimate",
    )
    ax2.fill_between(tau_grid, did_ci_lo, did_ci_hi, alpha=0.3, color="purple", label="95% CI")
except:
    # Fallback: use CQTE with SEs if DiD didn't work
    cqte_ci_lo = [cqte_results.get(tau, 0) - 1.96 * cqte_se.get(tau, 0) for tau in tau_grid]
    cqte_ci_hi = [cqte_results.get(tau, 0) + 1.96 * cqte_se.get(tau, 0) for tau in tau_grid]
    ax2.plot(
        tau_grid,
        cqte_vals,
        "o-",
        linewidth=2.5,
        markersize=8,
        color="purple",
        label="CQTE Estimate",
    )
    ax2.fill_between(tau_grid, cqte_ci_lo, cqte_ci_hi, alpha=0.3, color="purple", label="95% CI")

ax2.axhline(0, color="black", linestyle="--", linewidth=1.5, alpha=0.6, label="No Effect")
ax2.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Treatment Effect ($)", fontsize=12, fontweight="bold")
ax2.set_title("Panel B: QTE with Confidence Intervals", fontsize=14, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "09_qte_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"\nPlot saved to: {PLOTS_DIR / '09_qte_comparison.png'}")

## 4. Testing for Heterogeneous Treatment Effects

A crucial question: **Are treatment effects truly heterogeneous across quantiles, or is the variation just noise?**

We test:
- H0: QTE(\u03c4) = constant for all \u03c4 (homogeneous treatment effect)
- H1: QTE(\u03c4) varies with \u03c4 (heterogeneous treatment effect)

In [ ]:
# ============================================================
# HETEROGENEITY TESTS
# ============================================================

# Test 1: Built-in constant effects test (using RIF results)
print("TEST 1: Constant Effects Test (Built-in)")
print("=" * 50)

try:
    const_test = rif_result.test_constant_effects()
    print(f"Test statistic: {const_test['test_statistic']:.4f}")
    print(f"p-value: {const_test['p_value']:.4f}")
    if const_test["reject_constant"] is not None:
        if const_test["reject_constant"]:
            print("=> REJECT H0: Treatment effects are heterogeneous across quantiles")
        else:
            print("=> FAIL TO REJECT H0: No significant evidence of heterogeneity")
except Exception as e:
    print(f"Could not run built-in test: {e}")

# Test 2: Manual pairwise test — QTE(0.1) vs QTE(0.9)
print("\n\nTEST 2: Pairwise Comparison QTE(0.1) vs QTE(0.9)")
print("=" * 50)

try:
    # Use RIF results (which have standard errors)
    qte_low = rif_result.qte_results[0.1]["qte"]
    qte_high = rif_result.qte_results[0.9]["qte"]
    se_low = rif_result.qte_results[0.1]["se"]
    se_high = rif_result.qte_results[0.9]["se"]

    diff = qte_high - qte_low
    se_diff = np.sqrt(se_low**2 + se_high**2)  # Conservative (ignores covariance)
    t_stat = diff / se_diff
    p_value = 2 * (1 - stats.norm.cdf(abs(t_stat)))

    print("H0: QTE(0.1) = QTE(0.9)")
    print(f"QTE(0.1) = ${qte_low:.0f}")
    print(f"QTE(0.9) = ${qte_high:.0f}")
    print(f"Difference = ${diff:.0f}")
    print(f"SE(diff) = ${se_diff:.0f}")
    print(f"t-statistic = {t_stat:.2f}")
    print(f"p-value = {p_value:.4f}")

    if p_value < 0.05:
        print("\n=> REJECT H0: Significant heterogeneity in treatment effects")
        if diff < 0:
            print("   QTE(0.1) > QTE(0.9) => PROGRESSIVE program (helps low earners more)")
        else:
            print("   QTE(0.1) < QTE(0.9) => REGRESSIVE program (helps high earners more)")
    else:
        print("\n=> FAIL TO REJECT H0: No significant evidence of heterogeneity")
except Exception as e:
    print(f"Pairwise test error: {e}")

# Test 3: Visual inspection
print("\n\nTEST 3: Summary of QTE across quantiles")
print("=" * 50)
try:
    rif_df = rif_result.to_dataframe()
    print(f"Mean QTE: ${rif_df['qte'].mean():.0f}")
    print(f"Std of QTE: ${rif_df['qte'].std():.0f}")
    print(f"Range: ${rif_df['qte'].min():.0f} to ${rif_df['qte'].max():.0f}")
    print(f"Coefficient of variation: {rif_df['qte'].std() / abs(rif_df['qte'].mean()):.2%}")
except:
    pass

## 5. Case Study: Progressive vs Regressive Programs

### Policy Interpretation of QTE Patterns

```
PROGRESSIVE PROGRAM (QTE decreasing in τ):
  QTE(0.1) > QTE(0.5) > QTE(0.9)
  → Reduces inequality
  → Targets those most in need
  → Example: Basic income guarantee, job training for unskilled

REGRESSIVE PROGRAM (QTE increasing in τ):
  QTE(0.1) < QTE(0.5) < QTE(0.9)
  → May increase inequality
  → Benefits high earners more
  → Example: R&D tax credits, professional development subsidies

NEUTRAL PROGRAM (QTE flat):
  QTE(τ) ≈ constant for all τ
  → Proportional effect across distribution
  → ATE is a sufficient summary
```

Let's simulate both types and compare:

In [ ]:
# ============================================================
# SIMULATING PROGRESSIVE vs REGRESSIVE PROGRAMS
# ============================================================

np.random.seed(42)
n_sim = 1000

# Base earnings (log-normal)
base_ability = np.random.normal(0, 1, n_sim)
base_earnings = 3000 + 500 * base_ability + np.random.normal(0, 300, n_sim)
treatment_sim = np.random.binomial(1, 0.5, n_sim)

# Program A: PROGRESSIVE (helps low earners more)
te_progressive = 800 - 600 * stats.norm.cdf(base_ability)  # TE decreases with ability
earnings_A = base_earnings + treatment_sim * te_progressive

# Program B: REGRESSIVE (helps high earners more)
te_regressive = 200 + 600 * stats.norm.cdf(base_ability)  # TE increases with ability
earnings_B = base_earnings + treatment_sim * te_regressive

# Estimate QTE for each program
tau_fine = np.arange(0.05, 0.96, 0.05)

qte_A = []
qte_B = []
for tau in tau_fine:
    q1_A = np.quantile(earnings_A[treatment_sim == 1], tau)
    q0_A = np.quantile(earnings_A[treatment_sim == 0], tau)
    qte_A.append(q1_A - q0_A)

    q1_B = np.quantile(earnings_B[treatment_sim == 1], tau)
    q0_B = np.quantile(earnings_B[treatment_sim == 0], tau)
    qte_B.append(q1_B - q0_B)

ate_A = earnings_A[treatment_sim == 1].mean() - earnings_A[treatment_sim == 0].mean()
ate_B = earnings_B[treatment_sim == 1].mean() - earnings_B[treatment_sim == 0].mean()

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Program A
ax1.plot(tau_fine, qte_A, "o-", color="#009E73", linewidth=2, markersize=5, label="QTE(\u03c4)")
ax1.axhline(ate_A, color="red", linestyle="--", linewidth=2, label=f"ATE = ${ate_A:.0f}")
ax1.axhline(0, color="black", linestyle=":", alpha=0.4)
ax1.fill_between(tau_fine, qte_A, ate_A, alpha=0.1, color="green")
ax1.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Treatment Effect ($)", fontsize=12, fontweight="bold")
ax1.set_title("Program A: PROGRESSIVE\n(Helps low earners more)", fontsize=14, fontweight="bold")
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)
ax1.annotate(
    "Low earners gain MORE",
    xy=(0.1, qte_A[1]),
    fontsize=10,
    xytext=(0.3, qte_A[1] + 100),
    arrowprops={"arrowstyle": "->", "color": "green"},
)

# Program B
ax2.plot(tau_fine, qte_B, "o-", color="#D55E00", linewidth=2, markersize=5, label="QTE(\u03c4)")
ax2.axhline(ate_B, color="red", linestyle="--", linewidth=2, label=f"ATE = ${ate_B:.0f}")
ax2.axhline(0, color="black", linestyle=":", alpha=0.4)
ax2.fill_between(tau_fine, qte_B, ate_B, alpha=0.1, color="orange")
ax2.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Treatment Effect ($)", fontsize=12, fontweight="bold")
ax2.set_title("Program B: REGRESSIVE\n(Helps high earners more)", fontsize=14, fontweight="bold")
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)
ax2.annotate(
    "High earners gain MORE",
    xy=(0.85, qte_B[-3]),
    fontsize=10,
    xytext=(0.5, qte_B[-3] + 100),
    arrowprops={"arrowstyle": "->", "color": "darkorange"},
)

plt.tight_layout()
plt.show()

print(f"\nProgram A (Progressive): ATE = ${ate_A:.0f}")
print(f"  QTE(0.1) = ${qte_A[1]:.0f}, QTE(0.9) = ${qte_A[-2]:.0f}")
print(f"  => Low earners gain ${qte_A[1] - qte_A[-2]:.0f} MORE than high earners")

print(f"\nProgram B (Regressive): ATE = ${ate_B:.0f}")
print(f"  QTE(0.1) = ${qte_B[1]:.0f}, QTE(0.9) = ${qte_B[-2]:.0f}")
print(f"  => High earners gain ${qte_B[-2] - qte_B[1]:.0f} MORE than low earners")

print(
    f"\nBoth programs have similar ATEs (~${(ate_A + ate_B) / 2:.0f}), but very different distributional effects!"
)

## 6. When to Use Which Method

### Decision Guide

```
CHOOSE YOUR QTE METHOD:

Cross-sectional data + Random treatment?
  └─ YES → CQTE (Standard QR) or UQTE (RIF)
      ├─ Interested in conditional effects? → CQTE
      │    "Effect at τ-quantile, holding X constant"
      └─ Interested in population quantiles? → UQTE (RIF)
           "Effect on τ-quantile of overall distribution"

Panel data + Treatment varies over time?
  └─ YES → DiD-QR or CiC
      ├─ Parallel trends in quantiles plausible? → DiD-QR
      │    "Simpler, but requires parallel trends at each τ"
      └─ Parallel trends questionable? → CiC (more robust)
           "Uses full distributional information"

Selection on observables?
  └─ YES → Add controls to CQTE or use RIF regression

Selection on unobservables?
  └─ YES → Panel methods (DiD, CiC) or IV-QR
```

### Summary Table

In [ ]:
# ============================================================
# SUMMARY COMPARISON TABLE
# ============================================================

summary_data = {
    "Method": ["CQTE (Standard QR)", "UQTE (RIF)", "DiD-QR", "Changes-in-Changes"],
    "Type": ["Conditional", "Unconditional", "Panel DiD", "Panel DiD"],
    "Data Required": ["Cross-section", "Cross-section", "Panel (2 periods)", "Panel (2 periods)"],
    "Key Assumption": [
        "Selection on observables",
        "Selection on observables",
        "Parallel trends in quantiles",
        "Rank invariance",
    ],
    "Inference": ["Analytic/Bootstrap", "Robust SE", "Bootstrap", "Bootstrap (complex)"],
    "Captures": [
        "Conditional heterogeneity",
        "Population quantile shifts",
        "Time-differenced effects",
        "Distributional changes",
    ],
}

summary_df = pd.DataFrame(summary_data)
display(summary_df.style.set_caption("Comparison of QTE Methods"))

# Also summarize our results
print("\n\nOur Estimates Summary:")
print("=" * 65)
print(f"{'\u03c4':<6} {'CQTE':>10} {'UQTE':>10} {'DiD-QR':>10} {'CiC':>10}")
print("-" * 65)

for tau in tau_grid:
    row = f"{tau:<6.2f}"

    # CQTE
    val = cqte_results.get(tau, np.nan)
    row += f" {val:>10.1f}" if not np.isnan(val) else f" {'N/A':>10}"

    # UQTE
    try:
        val = rif_result.qte_results[tau]["qte"]
        row += f" {val:>10.1f}"
    except:
        row += f" {'N/A':>10}"

    # DiD-QR
    try:
        val = did_result.qte_results[tau]["qte"]
        row += f" {val:>10.1f}"
    except:
        row += f" {'N/A':>10}"

    # CiC
    try:
        val = cic_result.qte_results[tau]["qte"]
        row += f" {val:>10.1f}"
    except:
        row += f" {'N/A':>10}"

    print(row)

## 7. Exercises

---

### Exercise 1: CQTE vs UQTE (Easy)

**Task**: Explain in your own words:
1. What is the difference between CQTE and UQTE?
2. Give an example where CQTE and UQTE would give very different answers.
3. Which is more relevant for policy evaluation? Why?

In [ ]:
# Exercise 1: CQTE vs UQTE (Easy)
# Write your answers as comments or print statements below:

# 1. Difference between CQTE and UQTE:
# YOUR ANSWER HERE

# 2. Example where they differ:
# YOUR ANSWER HERE

# 3. Which is more relevant for policy?
# YOUR ANSWER HERE

### Exercise 2: Manual RIF Regression (Easy)

**Task**: Implement RIF regression from scratch:
1. Compute the sample quantile Q_\u03c4 for \u03c4 = 0.5
2. Estimate the density f_Y(Q_\u03c4) using a Gaussian kernel
3. Compute the RIF for each observation
4. Regress RIF on the treatment indicator
5. Compare your result with the PanelBox UQTE estimate

In [ ]:
# Exercise 2: Manual RIF Regression (Easy)

tau_ex2 = 0.5
y_ex2 = data["earnings"].values
d_ex2 = data["treatment"].values

# Step 1: Compute sample quantile
# Q_tau = ???

# Step 2: Estimate density at quantile (Gaussian kernel)
# bandwidth = ???  (hint: Silverman's rule = 1.06 * std(y) * n^(-1/5))
# f_tau = ???

# Step 3: Compute RIF
# RIF = Q_tau + (tau - I(y <= Q_tau)) / f_tau

# Step 4: Regress RIF on treatment (use np.linalg.lstsq or statsmodels)
# X_rif = np.column_stack([np.ones(len(y_ex2)), d_ex2])
# beta_rif = ???

# Step 5: Compare with PanelBox
# print(f"Manual UQTE: ${beta_rif[1]:.2f}")
# print(f"PanelBox UQTE: ${rif_result.qte_results[0.5]['qte']:.2f}")

### Exercise 3: Testing Parallel Trends (Medium)

**Task**: The DiD-QR method assumes parallel trends in quantiles. Test this using pre-treatment data:
1. Split the pre-treatment period into two sub-periods (or use the pre-period data)
2. Compare quantile trends between treated and control groups
3. Plot the quantile paths and visually assess parallel trends
4. Discuss: If parallel trends fails at some quantiles but not others, what does this mean?

In [ ]:
# Exercise 3: Testing Parallel Trends (Medium)

# Use the pre-treatment data to check if treated and control groups
# had similar quantile trends before the program

# Hint: Compare quantiles of treated vs control in period=0
# If the distributions are "parallel" (same shape, different level),
# DiD-QR assumptions are more plausible

pre_data = data[data["period"] == 0]

# TODO: Compute quantiles for treated and control groups in pre-period
# TODO: Plot and compare
# TODO: Discuss implications for DiD-QR validity

### Exercise 4: ATE Hides Heterogeneity (Medium)

**Task**: Simulate data where:
1. Low earners (below median) gain $1000 from treatment
2. High earners (above median) LOSE $200 from treatment
3. Show that ATE is positive (~$400) but misleading
4. Compute QTE at \u03c4 = {0.1, 0.25, 0.5, 0.75, 0.9} to reveal the true pattern
5. Discuss: What policy conclusion would you reach with ATE alone vs QTE?

In [ ]:
# Exercise 4: ATE Hides Heterogeneity (Medium)

np.random.seed(42)
n_ex4 = 2000

# TODO: Generate base earnings (e.g., log-normal)
# TODO: Generate treatment assignment (random)
# TODO: Create heterogeneous treatment effects:
#        - Large positive for low earners
#        - Negative for high earners
# TODO: Compute ATE and QTE
# TODO: Show that ATE is misleading
# TODO: Plot QTE(τ) vs ATE

### Exercise 5: DiD-QR vs Changes-in-Changes (Hard)

**Task**: Simulate data where parallel trends in quantiles FAILS:
1. Generate two groups with different variance evolution over time
2. Apply DiD-QR and CiC methods
3. Show that DiD-QR gives biased results while CiC is more robust
4. Discuss when each method is preferred

*Hint*: Let the control group's variance increase more than the treated group's, violating parallel trends at extreme quantiles.

In [ ]:
# Exercise 5: DiD-QR vs CiC (Hard)

np.random.seed(42)
n_ex5 = 500

# TODO: Simulate data with non-parallel trends in quantiles
# Hint: Let variance change differently for treated vs control

# Pre-period
# y_treated_pre = np.random.normal(5000, 500, n_ex5)
# y_control_pre = np.random.normal(5000, 500, n_ex5)

# Post-period (variance changes differently)
# y_control_post = np.random.normal(5200, 800, n_ex5)  # variance increases
# y_treated_post = np.random.normal(5500, 500, n_ex5)  # variance stays same + treatment effect

# TODO: Construct panel DataFrame
# TODO: Apply DiD-QR and CiC
# TODO: Compare estimates and discuss which is more reliable

### Exercise 6: Minimum Wage Policy Analysis (Hard)

**Task**: Simulate a minimum wage increase scenario:
1. Generate a realistic wage distribution (log-normal)
2. Simulate a minimum wage increase from $1500 to $1800
3. Model the effects:
   - Workers below new minimum get a raise (positive QTE at low quantiles)
   - Some workers above minimum may face reduced hours (negative QTE at medium quantiles)
   - High earners are unaffected (QTE \u2248 0 at high quantiles)
4. Estimate QTE and discuss:
   - Does the policy help the intended beneficiaries?
   - Are there unintended negative effects?
   - What would ATE alone tell us vs the full QTE picture?

In [ ]:
# Exercise 6: Minimum Wage Policy Analysis (Hard)

np.random.seed(42)

# TODO: Generate pre-treatment wage distribution
# TODO: Simulate minimum wage increase effects
# TODO: Estimate QTE across the wage distribution
# TODO: Create a comprehensive plot showing:
#   - QTE(τ) with confidence bands
#   - ATE reference line
#   - Annotation of policy effects at different quantiles
# TODO: Write policy conclusions

## Summary

### Key Takeaways

1. **ATE is not enough**: When treatment effects are heterogeneous, QTE reveals the full picture
2. **CQTE vs UQTE**: Conditional effects (within strata) vs unconditional effects (population level) --- use UQTE for policy evaluation
3. **DiD-QR**: Extends standard DiD to quantiles --- requires parallel trends at each quantile
4. **Changes-in-Changes**: More robust than DiD-QR --- uses full distributional information
5. **Heterogeneity testing**: Always test whether QTE varies significantly across quantiles
6. **Policy design**: Progressive programs (QTE decreasing) reduce inequality; regressive programs (QTE increasing) may worsen it

### References

1. **Firpo, S., Fortin, N. M., & Lemieux, T. (2009)**. "Unconditional Quantile Regressions". *Econometrica*, 77(3), 953-973.
2. **Athey, S., & Imbens, G. W. (2006)**. "Identification and Inference in Nonlinear Difference-in-Differences Models". *Econometrica*, 74(2), 431-497.
3. **Callaway, B., & Li, T. (2019)**. "Quantile Treatment Effects in Difference in Differences Models with Panel Data". *Quantitative Economics*, 10(4), 1579-1618.
4. **Abadie, A., Angrist, J., & Imbens, G. (2002)**. "Instrumental Variables Estimates of the Effect of Subsidized Training on the Quantiles of Trainee Earnings". *Econometrica*, 70(1), 91-117.

### Next Steps

- **Notebook 10**: Instrumental variables quantile regression
- Advanced: Quantile structural equation models
- Applied: Wage decomposition using RIF regression